In [ ]:
!pip install -q pandas numpy hnswlib tqdm pyarrow sentence-transformers

In [ ]:
import os
import re
import json
import numpy as np
import pandas as pd
import hnswlib
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from google.colab import drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# CONFIG
# ============================================================

ARTIFACT_DIR = "data/raw"  # change if needed

META_PATH = f"{ARTIFACT_DIR}/meta.parquet"
EMB_PATH = f"{ARTIFACT_DIR}/embeddings.npy"
INDEX_PATH = f"{ARTIFACT_DIR}/hnsw_cosine.bin"

OUTPUT_DIR = "data/intermediate/topic_candidates"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TEXT_COL = "comment_text"

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # use same model used to build embeddings if possible

TOP_K_HNSW_PER_QUERY = 20000
MAX_ROWS_PER_TOPIC = 50000
MIN_SIMILARITY = 0.25

In [ ]:
# ============================================================
# YOUR KEYWORD PATTERNS
# ============================================================

BLM_PATTERNS = [
    r"\bblack lives matter\b",
    r"\bblm\b",
    r"\bgeorge floyd\b",
    r"\bpolice brutality\b",
    r"\bdefund the police\b",
    r"\bracial justice\b",
    r"\bsystemic racism\b",
    r"\bracial inequality\b",
]

GUN_PATTERNS = [
    r"\bgun\b",
    r"\bguns\b",
    r"\bgun control\b",
    r"\bgun rights\b",
    r"\bgun legalization\b",
    r"\bfirearm\b",
    r"\bfirearms\b",
    r"\bsecond amendment\b",
    r"\b2nd amendment\b",
    r"\bnra\b",
    r"\bconcealed carry\b",
    r"\bopen carry\b",
    r"\bassault weapon\b",
    r"\bbackground checks\b",
    r"\bgun ban\b",
]

MARRIAGE_PATTERNS = [
    r"\bsame[- ]sex marriage\b",
    r"\bgay marriage\b",
    r"\bmarriage equality\b",
    r"\bequal marriage\b",
    r"\bcivil union\b",
    r"\bhomosexual marriage\b",
    r"\blgbt marriage\b",
]

TOPICS = {
    "BLM": {
        "patterns": BLM_PATTERNS,
        "queries": [
            "Black Lives Matter movement",
            "BLM protests and racial justice",
            "George Floyd and police brutality",
            "defund the police and systemic racism",
            "opinions about racial inequality and police violence",
        ],
    },
    "guns": {
        "patterns": GUN_PATTERNS,
        "queries": [
            "gun control debate",
            "gun rights and second amendment",
            "firearms ownership and gun laws",
            "NRA and gun policy",
            "opinions about background checks and assault weapons",
        ],
    },
    "same_sex_marriage": {
        "patterns": MARRIAGE_PATTERNS,
        "queries": [
            "same sex marriage",
            "gay marriage legalization",
            "marriage equality",
            "LGBT marriage rights",
            "civil unions and equal marriage",
        ],
    },
}

In [ ]:
# ============================================================
# LOAD DATA + INDEX
# ============================================================

print("Loading metadata...")
meta = pd.read_parquet(META_PATH)

print("Loading embeddings...")
emb = np.load(EMB_PATH, mmap_mode="r")
n, dim = emb.shape

assert len(meta) == n, "meta.parquet and embeddings.npy must have same number of rows"

print("Rows:", n)
print("Embedding dim:", dim)

print("Loading HNSW index...")
index = hnswlib.Index(space="cosine", dim=dim)
index.load_index(INDEX_PATH, max_elements=n)
index.set_ef(500)

print("Loading embedding model...")
model = SentenceTransformer(MODEL_NAME)

print("Done.")

In [ ]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def compile_topic_regex(patterns):
    return re.compile("|".join(patterns), flags=re.IGNORECASE)


def keyword_match_ids(df, text_col, patterns):
    regex = compile_topic_regex(patterns)

    texts = df[text_col].fillna("").astype(str)

    mask = texts.str.contains(regex, regex=True, na=False)
    ids = df.index[mask].to_numpy(dtype=np.int64)

    return ids


def hnsw_query_ids(topic_queries, top_k=20000, min_similarity=0.25):
    all_ids = []
    all_scores = []

    for query in topic_queries:
        q_emb = model.encode(
            query,
            normalize_embeddings=True,
            convert_to_numpy=True
        ).astype(np.float32)

        labels, distances = index.knn_query(q_emb, k=top_k)

        labels = labels[0]
        distances = distances[0]

        # hnswlib cosine distance = 1 - cosine similarity
        similarities = 1.0 - distances

        keep = similarities >= min_similarity

        all_ids.extend(labels[keep].tolist())
        all_scores.extend(similarities[keep].tolist())

    if len(all_ids) == 0:
        return pd.DataFrame(columns=["row_id", "hnsw_similarity"])

    result = pd.DataFrame({
        "row_id": all_ids,
        "hnsw_similarity": all_scores
    })

    # Keep best score per row
    result = (
        result.sort_values("hnsw_similarity", ascending=False)
        .drop_duplicates("row_id")
        .reset_index(drop=True)
    )

    return result


def extract_topic_dataset(
    topic_name,
    topic_info,
    meta,
    text_col,
    top_k_hnsw_per_query=20000,
    min_similarity=0.25,
    max_rows_per_topic=50000,
):
    print(f"\n==============================")
    print(f"Extracting topic: {topic_name}")
    print(f"==============================")

    # 1. keyword matches
    kw_ids = keyword_match_ids(meta, text_col, topic_info["patterns"])
    kw_df = pd.DataFrame({
        "row_id": kw_ids,
        "keyword_match": True
    })

    print(f"Keyword matches: {len(kw_df):,}")

    # 2. semantic HNSW retrieval
    hnsw_df = hnsw_query_ids(
        topic_info["queries"],
        top_k=top_k_hnsw_per_query,
        min_similarity=min_similarity
    )

    print(f"HNSW semantic matches: {len(hnsw_df):,}")

    # 3. merge keyword + HNSW candidates
    merged_ids = pd.DataFrame({
        "row_id": np.unique(
            np.concatenate([
                kw_df["row_id"].to_numpy(dtype=np.int64) if len(kw_df) else np.array([], dtype=np.int64),
                hnsw_df["row_id"].to_numpy(dtype=np.int64) if len(hnsw_df) else np.array([], dtype=np.int64),
            ])
        )
    })

    merged = merged_ids.merge(kw_df, on="row_id", how="left")
    merged = merged.merge(hnsw_df, on="row_id", how="left")

    merged["keyword_match"] = merged["keyword_match"].fillna(False)
    merged["hnsw_similarity"] = merged["hnsw_similarity"].fillna(0.0)

    # 4. priority score
    # keyword matches are highly trusted; hnsw similarity helps rank semantic matches
    merged["topic_candidate_score"] = (
        merged["keyword_match"].astype(int) * 1.0
        + merged["hnsw_similarity"]
    )

    merged = merged.sort_values(
        ["topic_candidate_score", "hnsw_similarity"],
        ascending=False
    )

    if max_rows_per_topic is not None:
        merged = merged.head(max_rows_per_topic)

    # 5. collect rows
    topic_df = meta.iloc[merged["row_id"].to_numpy()].copy()
    topic_df["original_row_id"] = merged["row_id"].to_numpy()
    topic_df["topic"] = topic_name
    topic_df["keyword_match"] = merged["keyword_match"].to_numpy()
    topic_df["hnsw_similarity"] = merged["hnsw_similarity"].to_numpy()
    topic_df["topic_candidate_score"] = merged["topic_candidate_score"].to_numpy()

    # 6. optional text deduplication
    topic_df["_text_norm"] = (
        topic_df[text_col]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    before = len(topic_df)
    topic_df = topic_df.drop_duplicates("_text_norm").drop(columns=["_text_norm"])
    after = len(topic_df)

    print(f"After text deduplication: {after:,} rows removed {before-after:,}")

    return topic_df

In [ ]:
# ============================================================
# RUN EXTRACTION FOR ALL 3 TOPICS
# ============================================================

topic_dfs = {}

for topic_name, topic_info in TOPICS.items():
    topic_df = extract_topic_dataset(
        topic_name=topic_name,
        topic_info=topic_info,
        meta=meta,
        text_col=TEXT_COL,
        top_k_hnsw_per_query=TOP_K_HNSW_PER_QUERY,
        min_similarity=MIN_SIMILARITY,
        max_rows_per_topic=MAX_ROWS_PER_TOPIC,
    )

    out_path = f"{OUTPUT_DIR}/{topic_name}_candidates.csv"
    topic_df.to_csv(out_path, index=False, encoding="utf-8-sig")

    print(f"Saved: {out_path}")
    topic_dfs[topic_name] = topic_df

# ============================================================
# SUMMARY
# ============================================================

summary = []

for topic_name, df in topic_dfs.items():
    summary.append({
        "topic": topic_name,
        "rows": len(df),
        "keyword_matches": int(df["keyword_match"].sum()),
        "semantic_only": int((~df["keyword_match"]).sum()),
        "mean_hnsw_similarity": float(df["hnsw_similarity"].mean()),
        "max_hnsw_similarity": float(df["hnsw_similarity"].max()),
    })

summary_df = pd.DataFrame(summary)
summary_df

# ============================================================
# SAVE COMBINED FILE TOO
# ============================================================

combined = pd.concat(topic_dfs.values(), ignore_index=True)

combined_path = f"{OUTPUT_DIR}/all_three_topics_candidates.csv"
combined.to_csv(combined_path, index=False, encoding="utf-8-sig")

print("Saved combined file:", combined_path)
print("Total rows:", len(combined))